# 05 - Review minimal assemblies and run the technical baseline

Run this notebook after notebook 04 has downloaded, assembled, and quality-checked the four-isolate minimal cohort. It creates an assembly-backed manifest, generates 999-base genomic windows, and runs the character-k-mer baseline as a technical pipeline validation.

The smoke cohort is too small for a scientific performance claim. Its purpose is to verify the complete workflow before scaling.

## 1. Configuration

In [ ]:
from __future__ import annotations

from pathlib import Path
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SMOKE_PLAN_PATH = PROJECT_ROOT / "data" / "manifests" / "ena_assembly_smoke_plan.csv"
SPLITS_PATH = PROJECT_ROOT / "data" / "processed" / "isolate_splits.csv"
ASSEMBLY_DIR = PROJECT_ROOT / "data" / "raw" / "assemblies"
SMOKE_MANIFEST_PATH = PROJECT_ROOT / "data" / "manifests" / "isolate_manifest_smoke_assemblies.csv"
WINDOWS_PATH = PROJECT_ROOT / "data" / "processed" / "smoke_windows.csv"
METRICS_PATH = PROJECT_ROOT / "reports" / "smoke_kmer_baseline.json"
RUN_WINDOWS = True
RUN_BASELINE = True

print("Project root:", PROJECT_ROOT)

## 2. Inspect assembly availability and FASTA statistics

The notebook stops if any minimal-plan assembly is missing. Finish notebook 04 first rather than silently analyzing a partial cohort.

In [ ]:
smoke_plan = pd.read_csv(SMOKE_PLAN_PATH, dtype={"label": str}).fillna("")
smoke_ids = smoke_plan["isolate_id"].drop_duplicates().tolist()

def fasta_stats(path: Path) -> dict[str, int]:
    lengths = []
    sequence_length = 0
    with path.open() as handle:
        for line in handle:
            if line.startswith(">"):
                if sequence_length:
                    lengths.append(sequence_length)
                sequence_length = 0
            else:
                sequence_length += len(line.strip())
    if sequence_length:
        lengths.append(sequence_length)
    return {
        "contigs": len(lengths),
        "total_bp": sum(lengths),
        "longest_contig_bp": max(lengths, default=0),
    }

assembly_rows = []
for isolate_id in smoke_ids:
    assembly_path = ASSEMBLY_DIR / f"{isolate_id}.fasta"
    row = {"isolate_id": isolate_id, "assembly_path": str(assembly_path.relative_to(PROJECT_ROOT))}
    row["exists"] = assembly_path.exists()
    if assembly_path.exists():
        row.update(fasta_stats(assembly_path))
    assembly_rows.append(row)

assembly_audit = pd.DataFrame(assembly_rows)
print("Expected assemblies:", len(assembly_audit))
print("Available assemblies:", int(assembly_audit["exists"].sum()))
assembly_audit

## 3. Create the assembly-backed smoke manifest

Review the basic FASTA statistics and QUAST reports under `reports/assembly_qc/` before continuing.

In [ ]:
missing = assembly_audit.loc[~assembly_audit["exists"], "isolate_id"].tolist()
if missing:
    raise FileNotFoundError(f"Missing {len(missing)} minimal assemblies. Finish notebook 04 first: {missing}")

splits = pd.read_csv(SPLITS_PATH, dtype={"label": str}).fillna("")
smoke_manifest = splits[splits["isolate_id"].isin(smoke_ids)].copy()
smoke_manifest = smoke_manifest.drop(columns=["assembly_path"]).merge(
    assembly_audit[["isolate_id", "assembly_path"]],
    on="isolate_id",
    how="left",
    validate="one_to_one",
)
smoke_manifest.to_csv(SMOKE_MANIFEST_PATH, index=False)

print(pd.crosstab(smoke_manifest["split"], smoke_manifest["label"], margins=True))
print("Wrote:", SMOKE_MANIFEST_PATH.relative_to(PROJECT_ROOT))

## 4. Generate 999-base genomic windows

`RUN_WINDOWS` is enabled for this four-isolate technical cohort. The expected CSV footprint is approximately 13 MB.

In [ ]:
windows_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "build_windows.py"),
    "--manifest",
    str(SMOKE_MANIFEST_PATH),
    "--output",
    str(WINDOWS_PATH),
]
print(" ".join(windows_command))
if RUN_WINDOWS:
    subprocess.run(windows_command, cwd=PROJECT_ROOT, check=True)
if WINDOWS_PATH.exists():
    windows = pd.read_csv(WINDOWS_PATH)
    print("Windows:", len(windows))
    print("Isolates:", windows["isolate_id"].nunique())

## 5. Run the character-k-mer smoke baseline

`RUN_BASELINE` is enabled for this four-isolate technical cohort. Treat the resulting metrics as a pipeline check, not a model-performance result.

In [ ]:
baseline_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "train_kmer_baseline.py"),
    "--windows",
    str(WINDOWS_PATH),
    "--splits",
    str(SPLITS_PATH),
    "--output",
    str(METRICS_PATH),
]
print(" ".join(baseline_command))
if RUN_BASELINE:
    subprocess.run(baseline_command, cwd=PROJECT_ROOT, check=True)
if METRICS_PATH.exists():
    print(METRICS_PATH.read_text())

## 6. Next step

Once the four-isolate workflow is reproducible, scale cautiously. The current disk cannot hold the full 40.94 GB FASTQ cohort plus assembly intermediates. Use external storage, small sequential batches with cleanup, or a direct public-assembly retrieval strategy before running the full baseline. DNABERT2 comes after that conventional sequence baseline is stable.